# 2. 도구(Tools) & 메모리(Memory) 시스템

**exercise 2 | 도구 & 메모리 × APD 실습**

---

## 학습 목표
1. **로컬 도구**: Python 함수를 그대로 도구로 활용 — **Application Collector** 구현
2. **API 기반 도구**: 외부 서비스 연동 — **Essay Analyzer** (Wikipedia) + **Portfolio Evaluator** (Gemini Vision)
3. **플러그인 도구**: Gemini 내장 Google Search — **Essay Analyzer** 역량 확장
4. **메모리 시스템**: 단기(FIFO/요약) vs 장기(JSON 파일) 전략 비교

---

## 이번 exercise에서 구현할 APD 에이전트들

```
INPUT(지원서) → [Task 0: Application Collector 🟦] → [Task 1a: Essay Analyzer 🟧]
 ↑ 로컬 도구 ↑ API 도구 + 플러그인 도구
 [Task 1c: Portfolio Evaluator 🟧]
 ↑ API 도구 (Gemini Vision)
```

| 섹션 | 도구 유형 | 구현 에이전트 | APD Task |
|------|-----------|--------------|----------|
| 2 | 로컬 도구 | **Application Collector** | Task 0 |
| 4 | API 기반 도구 | **Essay Analyzer** (Wikipedia) | Task 1a |
| 4 | API 기반 도구 | **Portfolio Evaluator** (Gemini Vision) | Task 1c |
| 5 | 플러그인 도구 | **Essay Analyzer** + Google Search | Task 1a 확장 |

## 1. 왜 도구가 필요한가?

AI 모델은 텍스트를 생성하는 능력만 가집니다.
**도구(Tool)** 를 통해 모델이 실제 세계와 상호작용할 수 있습니다.

```
모델만 있을 때: 지원서 → 텍스트 생성 → 결과 (비결정론적 — 매번 결과가 달라질 수 있음)
도구가 있을 때: 지원서 → 도구 선택 → 도구 실행 → 결과 통합 → 결과 (결정론적 검증 가능)
```

### 예시: Application Collector에서 LLM만으로 유효성 검사를 하면?

| 질문 | LLM만 | 도구 사용 |
|------|-------|----------|
| 학번 형식이 맞는가? (8자리 숫자) | 매번 다른 판단 가능 | `validate_student_id()` → 항상 동일 |
| 이미 처리한 지원서인가? | 기억 없음 | `check_duplicate()` → DB 조회 |
| 전공명 표준화 | 비일관적 | `normalize_major()` → 규칙 적용 |

### 도구의 종류
- **로컬 도구**: 내부 로직/계산 (검증, 중복 확인, 데이터 정제)
- **API 기반 도구**: 외부 서비스 연동 (Wikipedia 검색, Gemini Vision)
- **플러그인 도구**: 플랫폼 내장 도구 활성화 (Google Search)

## 2. 환경 설정

In [1]:
# 필요한 라이브러리 설치
!pip install -q google-genai requests

from google import genai
from google.genai import types
from google.genai.types import HttpOptions
from google.colab import auth
import json
import os
import re
import requests
from datetime import datetime

# Vertex AI 인증 (Google 계정으로 로그인)
auth.authenticate_user()

PROJECT_ID = "scientific-management-494205"
LOCATION = "us-central1"

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
    http_options=HttpOptions(api_version="v1"),
)
print('✅ 환경 설정 완료')

✅ 환경 설정 완료


## 3. 로컬 도구 — Application Collector

### Application Collector의 역할

> 지원서를 수집하고, 필수 필드 유효성 검사 및 중복 제거를 수행합니다.
> **Method**: 스키마 검증 + 퍼지 매치 중복 제거

### 핵심 개념: 함수를 그대로 도구로 전달
Google Gen AI SDK는 Python 함수를 **그대로** 도구로 사용할 수 있습니다.
> **중요**: 함수의 **docstring** 이 모델이 도구를 이해하는 데 사용됩니다.
> 명확하고 구체적인 docstring을 작성하세요!

In [2]:
# ====================================================
# 로컬 도구 정의 — Application Collector용 3개 함수
# ====================================================

# 처리된 학번 추적 (세션 내 단기 메모리)
processed_ids: set = set()

# 전공명 표준화 사전
MAJOR_ALIASES = {
    '컴공': '컴퓨터공학과',
    '컴퓨터공학': '컴퓨터공학과',
    '전기전자': '전기공학과',
    '물리': '물리학과',
    '수학': '수학과',
    '통계': '통계학과',
    '소웨': '소프트웨어학과',
    '소프트웨어': '소프트웨어학과',
}


def validate_fields(name: str, student_id: str, major: str, courses: str) -> str:
    '''지원서 필수 필드의 존재 여부와 형식을 검사합니다.

    Args:
        name: 지원자 이름
        student_id: 학번 (8자리 숫자 형식, 예: 20240001)
        major: 전공명
        courses: 이수 완료 과목 목록 (쉼표 구분)

    Returns:
        검사 결과 메시지. 유효하면 "VALID", 오류가 있으면 오류 목록.
    '''
    errors = []

    if not name or not name.strip():
        errors.append('이름이 비어있습니다')

    if not student_id:
        errors.append('학번이 비어있습니다')
    elif not re.fullmatch(r'\d{8}', student_id.strip()):
        errors.append(f'학번 형식 오류: "{student_id}" (8자리 숫자 필요)')

    if not major or not major.strip():
        errors.append('전공이 비어있습니다')

    if errors:
        return '❌ 유효성 오류: ' + ' / '.join(errors)
    return f'✅ VALID: {name} ({student_id}, {major})'


def check_duplicate(student_id: str) -> str:
    '''이미 처리된 학번인지 확인하여 중복 지원을 방지합니다.

    Args:
        student_id: 확인할 학번 (8자리 숫자)

    Returns:
        "DUPLICATE" (이미 처리됨) 또는 "NEW" (신규 지원)
    '''
    sid = student_id.strip()
    if sid in processed_ids:
        return f'⚠️  DUPLICATE: 학번 {sid}는 이미 처리된 지원서입니다.'
    return f'✅ NEW: 학번 {sid}는 신규 지원자입니다.'


def normalize_application(name: str, student_id: str, major: str, courses: str) -> str:
    '''전공명 표준화 및 과목 목록 정제 후 정규화된 지원서를 반환합니다.

    Args:
        name: 지원자 이름
        student_id: 학번
        major: 전공명 (약칭 허용)
        courses: 이수 과목 목록 (쉼표 구분)

    Returns:
        정규화된 지원서 정보와 처리된 학번을 등록합니다.
    '''
    # 전공명 표준화
    normalized_major = MAJOR_ALIASES.get(major.strip(), major.strip())

    # 과목 목록 정제 (공백 제거, 중복 제거)
    course_list = [c.strip() for c in courses.split(',') if c.strip()]
    course_list = list(dict.fromkeys(course_list))  # 중복 제거 (순서 유지)

    # 학번 등록 (중복 방지용)
    processed_ids.add(student_id.strip())

    result = {
        'name': name.strip(),
        'student_id': student_id.strip(),
        'major': normalized_major,
        'courses': course_list,
        'processed_at': datetime.now().strftime('%Y-%m-%d %H:%M')
    }
    return f'✅ 정규화 완료: {json.dumps(result, ensure_ascii=False)}'


print('✅ 로컬 도구 3개 정의 완료')
print('  - validate_fields(): 필수 필드 유효성 검사')
print('  - check_duplicate(): 중복 지원 확인')
print('  - normalize_application(): 전공명 표준화 + 과목 정제')

✅ 로컬 도구 3개 정의 완료
  - validate_fields(): 필수 필드 유효성 검사
  - check_duplicate(): 중복 지원 확인
  - normalize_application(): 전공명 표준화 + 과목 정제


In [ ]:
# Application Collector 에이전트 생성
chat = client.chats.create(
    model='gemini-2.5-flash',
    config=types.GenerateContentConfig(
        tools=[validate_fields, check_duplicate, normalize_application],
        system_instruction='''
    당신은 동아리 지원서를 수집하고 검증하는 Application Collector 에이전트입니다.
    지원서를 받으면 반드시 이 순서로 처리하세요:
    1. check_duplicate()로 중복 지원 여부 확인
    2. validate_fields()로 필드 유효성 검사
    3. 유효한 경우 normalize_application()으로 데이터 정규화
    각 단계 결과를 명확히 보고하세요.
    ''',
        thinking_config=types.ThinkingConfig(thinking_budget=0)
    )
)

# 테스트 지원서 (다양한 케이스)
test_applications = [
    '이지은, 20240101, 컴공, 파이썬 기초,선형대수',       # 약칭 전공, 중복 과목
    '김철수, 2024ABC, 경영학과, 없음',                    # 학번 형식 오류
    '이지은, 20240101, 컴퓨터공학과, 파이썬 기초',        # 중복 지원자
    '박민준, 20240303, 물리, 선형대수',                   # 정상 신규 지원
]

for app in test_applications:
    print(f"\n{'='*55}")
    print(f'📥 지원서 입력: {app}')
    # 이름, 학번, 전공, 이수과목을 분리하여 전달
    parts = [p.strip() for p in app.split(',', 3)]
    name, sid, major = parts[0], parts[1], parts[2]
    courses = parts[3] if len(parts) > 3 else ''
    message = f'다음 지원서를 처리해줘: 이름={name}, 학번={sid}, 전공={major}, 이수과목={courses}'
    response = chat.send_message(message)
    print(f'🤖 Application Collector: {response.text}')


📥 지원서 입력: 이지은, 20240101, 컴공, 파이썬 기초,선형대수
🤖 Application Collector: ✅ 지원서가 성공적으로 처리되었습니다.

📥 지원서 입력: 김철수, 2024ABC, 경영학과, 없음
🤖 Application Collector: ✅ 지원서 처리에 실패했습니다. 학번 형식 오류가 발견되었습니다.

📥 지원서 입력: 이지은, 20240101, 컴퓨터공학과, 파이썬 기초
🤖 Application Collector: ⚠️ 학번 20240101은 이미 처리된 지원서이므로 추가 처리를 진행할 수 없습니다.

📥 지원서 입력: 박민준, 20240303, 물리, 선형대수
🤖 Application Collector: ✅ 지원서가 성공적으로 처리되었습니다.


## 4. 내부 동작 이해: Tool Calling 과정 들여다보기

에이전트가 도구를 어떻게 선택하고 실행하는지 단계별로 확인합니다.

In [4]:
# 자동 함수 호출 없이 직접 과정 확인
_manual_config = types.GenerateContentConfig(
    tools=[validate_fields, check_duplicate, normalize_application],
    system_instruction="동아리 지원서를 처리하는 Application Collector 에이전트입니다.",
    tool_config=types.ToolConfig(
        function_calling_config=types.FunctionCallingConfig(mode="ANY")
    )
)

messages = [types.Content(role="user", parts=[
    types.Part(text="이름=최수진, 학번=20240202, 전공=통계, 이수과목=파이썬 기초 를 처리해줘. 먼저 중복 확인부터 해줘.")
])]

# Step 1: 모델이 어떤 도구를 호출할지 결정
response = client.models.generate_content(model='gemini-2.5-flash-lite', contents=messages, config=_manual_config)

print("📋 Step 1: 모델의 도구 호출 결정")
function_call = None
for part in response.candidates[0].content.parts:
    if hasattr(part, "function_call") and part.function_call:
        fc = part.function_call
        function_call = fc
        print(f"  → 선택된 도구: {fc.name}")
        print(f"  → 파라미터: {dict(fc.args)}")

if function_call is None:
    print("  → 도구 호출 없음 (직접 응답)")
    print(f"  → 응답: {response.text}")
else:
    # Step 2: 도구 실제 실행
    print("🔧 Step 2: 도구 실제 실행")
    args = dict(function_call.args)
    tool_result = check_duplicate(args.get("student_id", "20240202"))
    print(f"  → 실행 결과: {tool_result}")

    # Step 3: 결과를 모델에 전달
    print("💬 Step 3: 최종 응답 생성")
    messages.append(response.candidates[0].content)
    messages.append(types.Content(
        role="tool",
        parts=[types.Part(function_response=types.FunctionResponse(
            name=function_call.name,
            response={"result": tool_result}
        ))]
    ))

    _final_config = types.GenerateContentConfig(
    tools=[validate_fields, check_duplicate, normalize_application],
    system_instruction="동아리 지원서를 처리하는 Application Collector 에이전트입니다."
    # tool_config 없음 = AUTO
)

    final_response = client.models.generate_content(model='gemini-2.5-flash-lite', contents=messages, config=_final_config)

    # 모델이 텍스트 대신 추가 도구를 호출할 수 있으므로 parts 확인
    final_parts = final_response.candidates[0].content.parts
    has_text = any(hasattr(p, "text") and p.text for p in final_parts)

    if has_text:
        print(f"  → 최종 응답: {final_response.text}")
    else:
        print("  → 모델이 추가 도구 호출 결정 (다음 단계 처리 필요)")
        for p in final_parts:
            if hasattr(p, "function_call") and p.function_call:
                print(f"     추가 호출: {p.function_call.name}({dict(p.function_call.args)})")

📋 Step 1: 모델의 도구 호출 결정
  → 선택된 도구: check_duplicate
  → 파라미터: {'student_id': '20240202'}
🔧 Step 2: 도구 실제 실행
  → 실행 결과: ⚠️  DUPLICATE: 학번 20240202는 이미 처리된 지원서입니다.
💬 Step 3: 최종 응답 생성
  → 최종 응답: 지원서 처리에 실패했습니다. 학번 20240202는 이미 처리된 지원서입니다.


## 5. API 기반 도구 — Essay Analyzer & Portfolio Evaluator

### 로컬 도구 vs API 기반 도구

| | 로컬 도구 | API 기반 도구 |
|---|---|---|
| **구현** | 직접 Python 함수 작성 | 외부 서비스 HTTP 호출 |
| **인증** | 불필요 | API Key 또는 OAuth |
| **예시** | validate_fields, check_duplicate | Wikipedia REST, Gemini Vision |
| **에이전트** | Application Collector | Essay Analyzer, Portfolio Evaluator |

---

### 5.1 Essay Analyzer + Wikipedia API (Task 1a)

> **Essay Analyzer**: 자기소개서에서 언급된 기술 개념을 Wikipedia로 조회하여
> 지원자의 기술 이해도를 평가합니다.

**Wikipedia API**: 인증 없음 — 가장 단순한 REST API 패턴

In [5]:
# Wikipedia API 도구 정의
def search_wikipedia(query: str, max_chars: int = 500) -> str:
    '''Wikipedia에서 기술 개념이나 용어를 검색합니다.

    Args:
        query: 검색할 키워드 (한국어 또는 영어)
        max_chars: 반환할 최대 문자 수 (기본 500)

    Returns:
        검색 결과 요약 텍스트. 찾지 못한 경우 오류 메시지.
    '''
    try:
        encoded_query = requests.utils.quote(query)
        url = f'https://ko.wikipedia.org/api/rest_v1/page/summary/{encoded_query}'
        headers = {'User-Agent': 'LET-Project/1.0 (educational; python-requests)'}
        response = requests.get(url, timeout=5, headers=headers)
        if response.status_code == 200:
            data = response.json()
            extract = data.get('extract', '정보를 찾을 수 없습니다.')
            return extract[:max_chars] + ('...' if len(extract) > max_chars else '')
        # 한국어 없으면 영어로 재시도
        url_en = f'https://en.wikipedia.org/api/rest_v1/page/summary/{encoded_query}'
        response_en = requests.get(url_en, timeout=5, headers=headers)
        if response_en.status_code == 200:
            data_en = response_en.json()
            extract_en = data_en.get('extract', '')
            return extract_en[:max_chars] + ('...' if len(extract_en) > max_chars else '')
        return f'\'{query}\' 관련 Wikipedia 정보를 찾을 수 없습니다.'
    except Exception as e:
        return f'검색 중 오류 발생: {str(e)}'


# 단독 테스트
print('Wikipedia 도구 테스트:')
print(search_wikipedia('머신러닝')[:300], '...')
print()
print(search_wikipedia('transformer model')[:300], '...')

Wikipedia 도구 테스트:
기계 학습(機械學習) 또는 머신 러닝(영어: machine learning, ML)은 경험을 통해 자동으로 개선하는 컴퓨터 알고리즘의 연구이다. 방대한 데이터를 분석해 '미래를 예측하는 기술'이자 인공지능의 한 분야로 간주된다. 기계 학습은 복잡한 패턴에 대한 학습을 통해 상황에 대한 예측과 의사 결정을 돕는다. 컴퓨터가 학습할 수 있도록 하는 알고리즘과 기술을 개발하는 분야이다. 가령, 기계 학습을 통해서 수신한 이메일이 스팸인지 아닌지를 구분할 수 있도록 훈련할 수 있다. ...

In deep learning, the transformer is a family of artificial neural network architectures based on the multi-head attention mechanism, in which text is converted to numerical representations called tokens, and each token is converted into a vector via lookup from a word embedding table. At each layer ...


In [6]:
# Essay Analyzer 에이전트 생성 (Wikipedia 도구 사용)
chat_essay = client.chats.create(
    model='gemini-2.5-flash',
    config=types.GenerateContentConfig(
        tools=[search_wikipedia],
        system_instruction='''
    당신은 동아리 지원자의 자기소개서를 심사하는 Essay Analyzer 에이전트입니다.

    심사 방법:
    1. 에세이에서 기술 용어/개념을 파악합니다.
    2. search_wikipedia()로 해당 개념을 조회합니다.
    3. 지원자의 설명과 Wikipedia 정보를 비교하여 이해도를 평가합니다.

    평가 항목:
    - 기술 이해도 (1~5점): 개념을 올바르게 이해하고 있는가?
    - 동아리 적합성 (1~5점): AI/ML 관련 관심이 드러나는가?
    - 글쓰기 명확성 (1~5점): 논리적으로 서술되었는가?
    - 총점 및 종합 의견
    ''',
        thinking_config=types.ThinkingConfig(thinking_budget=0)
    )
)

# 테스트 에세이
essay_samples = [
    '''
    지원자: 이지은
    자기소개서: 저는 머신러닝과 딥러닝에 관심이 많습니다. 특히 transformer model의 attention 메커니즘이
    언어 처리에 혁신을 가져왔다고 생각합니다. 파이썬으로 간단한 신경망을 구현해본 경험이 있으며,
    동아리에서 더 깊이 공부하고 싶습니다.
    ''',
    '''
    지원자: 박민준
    자기소개서: AI가 요즘 많이 쓰인다고 해서 지원했습니다. 컴퓨터 게임을 좋아하고
    코딩도 조금 해봤습니다. 열심히 하겠습니다.
    '''
]

for essay in essay_samples:
    print(f"\n{'='*55}")
    print(f'📄 에세이 제출: {essay.strip()[:60]}...')
    response = chat_essay.send_message(f'다음 에세이를 심사해주세요:\n{essay}')
    print(f'🤖 Essay Analyzer:\n{response.text}')


📄 에세이 제출: 지원자: 이지은
    자기소개서: 저는 머신러닝과 딥러닝에 관심이 많습니다. 특히 transformer m...
🤖 Essay Analyzer:
이지은 지원자 심사 결과:

**기술 이해도 (4/5점):**
지원자는 transformer model과 attention 메커니즘을 정확하게 언급하며, 이들이 언어 처리에 혁신을 가져왔다는 점을 인지하고 있습니다. Wikipedia 검색 결과와 비교했을 때, 개념에 대한 기본적인 이해는 충분한 것으로 보입니다. 다만, 파이썬으로 "간단한 신경망"을 구현했다는 언급만으로는 딥러닝 모델에 대한 깊이 있는 이해를 판단하기 어렵습니다.

**동아리 적합성 (5/5점):**
머신러닝과 딥러닝에 대한 높은 관심을 명확히 드러내고 있으며, 특히 transformer model과 attention 메커니즘을 언급한 점에서 AI/ML 분야에 대한 구체적인 흥미를 엿볼 수 있습니다. 동아리에서 더 깊이 공부하고 싶다는 의지를 표현하여 동아리 활동에 대한 열정이 기대됩니다.

**글쓰기 명확성 (4/5점):**
간결하고 명확하게 자신의 관심 분야와 경험, 동아리 지원 동기를 잘 전달하고 있습니다. 문장의 연결도 자연스러우며, 논리적인 흐름을 가지고 있습니다. 다만, "간단한 신경망"에 대한 구체적인 설명이 있었다면 더욱 좋았을 것입니다.

**총점: 13/15점**

**종합 의견:**
이지은 지원자는 머신러닝과 딥러닝 분야에 대한 높은 관심과 기본적인 이해를 갖추고 있습니다. 특히 transformer model과 attention 메커니즘을 언급한 점에서 최신 기술 트렌드에 대한 인지도도 보입니다. 파이썬을 활용한 신경망 구현 경험은 긍정적인 요소이지만, 어떤 종류의 신경망을 구현했는지 구체적으로 언급했다면 기술 이해도를 더 명확하게 평가할 수 있었을 것입니다. 동아리 활동을 통해 더욱 심도 있는 학습을 하고자 하는 의지가 강하여, 동아리에 긍정적인 기여를 할 수 있을 것으로 기대됩니다.

📄 에세이 제출: 지

### 5.2 Portfolio Evaluator + Gemini Vision API (Task 1c)

> **Portfolio Evaluator**: 지원자가 업로드한 포트폴리오 이미지를 Gemini Vision으로 분석하여
> 작업물의 품질, 관련성, 완성도를 평가합니다.

**Gemini Vision API**: Vertex AI를 통해 이미지 분석 가능 (별도 인증 불필요)

In [7]:
# Gemini Vision API 도구 정의
def analyze_portfolio_image(image_url: str, evaluation_rubric: str = '') -> str:
    '''지원자의 포트폴리오 이미지를 Gemini Vision으로 분석합니다.

    Args:
        image_url: 분석할 포트폴리오 이미지의 URL
        evaluation_rubric: 평가 기준 (예: "코드 품질, 시각화 완성도, AI 관련성")

    Returns:
        이미지 분석 결과와 포트폴리오 평가 내용.
    '''
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (compatible; LET-Project/1.0; python-requests)'}
        resp = requests.get(image_url, timeout=10, headers=headers)
        resp.raise_for_status()

        # MIME 타입 감지 (PIL 불필요)
        mime_type = resp.headers.get('Content-Type', 'image/jpeg').split(';')[0].strip()
        if not mime_type.startswith('image/'):
            mime_type = 'image/jpeg'

        image_part = types.Part.from_bytes(data=resp.content, mime_type=mime_type)

        rubric_text = evaluation_rubric if evaluation_rubric else '작업물의 품질, 완성도, AI/기술 관련성'
        question = f'''이 포트폴리오 이미지를 동아리 지원자의 작업물로서 평가해주세요.
평가 기준: {rubric_text}
각 기준을 1~5점으로 평가하고 종합 의견을 제시해주세요.'''

        result = client.models.generate_content(model='gemini-2.5-flash-lite', contents=[question, image_part])
        return result.text
    except Exception as e:
        return f'포트폴리오 분석 중 오류 발생: {str(e)}'


# 단독 테스트 — 공개 예시 이미지 사용
print('Gemini Vision 도구 테스트:')
# 데이터 시각화 예시 이미지 (source: https://www.scaler.com/blog/data-science-portfolio/)
sample_image_url = 'https://scaler-blog-prod-wp-content.s3.ap-south-1.amazonaws.com/wp-content/uploads/2024/08/22163741/Maggie-Wolff-1024x501.png'
print(analyze_portfolio_image(sample_image_url, '시각적 완성도, AI/기술 관련성'))

Gemini Vision 도구 테스트:
## 포트폴리오 평가

**지원자:** Maggie Wolff
**지원 분야:** (추정) 데이터 과학 관련 동아리

### 평가 기준

**1. 시각적 완성도:** 4/5점
* **강점:**
    * 전체적으로 깔끔하고 정돈된 디자인입니다.
    * 프로필 사진, 이름, 직책, 연락처 정보 등이 명확하게 배치되어 있습니다.
    * 글자 크기와 줄 간격이 적절하여 가독성이 좋습니다.
    * 링크(LinkedIn, Github)가 명확하게 표시되어 있어 접근성이 좋습니다.
* **개선점:**
    * 약간의 여백 조정이나 폰트 스타일 변경으로 조금 더 세련된 느낌을 줄 수 있습니다. (필수는 아님)

**2. AI/기술 관련성:** 5/5점
* **강점:**
    * "Data Scientist", "Product Analytics", "Travel Tech industry"와 같은 키워드를 통해 지원자의 전문 분야를 명확히 알 수 있습니다.
    * "recommender model", "advanced analysis", "A/B (hypothesis) testing", "predictive modeling" 등 AI 및 데이터 분석 관련 구체적인 기술 및 업무 경험을 언급하고 있습니다.
    * "MS in Data Science" 학위 소지자임을 명시하여 기술적 배경을 뒷받침합니다.
    * "Women in Data Science (WiDS)" 앰버서더 및 "Data Angels" 멘토링 프로그램 운영 경험은 관련 커뮤니티 활동 및 리더십을 보여줍니다.
    * "knowledge sharing"에 대한 언급은 동아리 활동에 긍정적인 영향을 줄 수 있는 부분입니다.
* **개선점:**
    * (만약 가능하다면) 언급된 프로젝트나 경험에 대한 더 구체적인 설명이나 링크를 추가하면 더욱 강력한 포트폴리오가 될 수 있습니다. (예: github 링크에 실제 프로젝트 코드 포함 등)

### 종합

In [8]:
# Portfolio Evaluator 에이전트 생성
chat_portfolio = client.chats.create(
    model='gemini-2.5-flash',
    config=types.GenerateContentConfig(
        tools=[analyze_portfolio_image],
        system_instruction='''
    당신은 동아리 지원자의 포트폴리오를 평가하는 Portfolio Evaluator 에이전트입니다.

    포트폴리오 평가 기준:
    - 기술 수준 (1~5점): 작업물의 기술적 완성도
    - AI/ML 관련성 (1~5점): 동아리 활동과 관련된 기술인가?
    - 창의성 (1~5점): 독창적인 아이디어나 접근법이 있는가?
    - 총점 및 합격 권고 여부 (권고/비권고)

    이미지 URL이 주어지면 analyze_portfolio_image() 도구를 사용하여 분석하세요.
    '''
    )
)

# 포트폴리오 이미지 URL로 평가 요청 (img source: https://www.scaler.com/blog/data-science-portfolio/)
portfolio_requests = [
    {
        'applicant': '이지은',
        'url': 'https://scaler-blog-prod-wp-content.s3.ap-south-1.amazonaws.com/wp-content/uploads/2024/08/22162613/Ger-Inberg-1024x576.png',
        'description': '데이터 분석 프로젝트 스크린샷'
    },
]

for req in portfolio_requests:
    print(f"\n{'='*55}")
    print(f'📁 지원자: {req["applicant"]} | 포트폴리오: {req["description"]}')
    message = f'{req["applicant"]} 지원자의 포트폴리오를 평가해주세요. 이미지 URL: {req["url"]}'
    response = chat_portfolio.send_message(message)
    print(f'🤖 Portfolio Evaluator:\n{response.text}')


📁 지원자: 이지은 | 포트폴리오: 데이터 분석 프로젝트 스크린샷
🤖 Portfolio Evaluator:
이지은 지원자의 포트폴리오 평가 결과입니다.

**평가 기준:**

*   **기술 수준:** 4/5
*   **AI/ML 관련성:** 3/5
*   **창의성:** 3/5

**총점:** 10/15점

---

### 상세 평가:

**1. 기술 수준 (4/5)**
다양한 종류의 데이터 시각화(지리 정보, 막대 그래프, 원형 차트, 선 그래프, 히스토그램 등)를 능숙하게 다루며, 복잡한 데이터를 이해하기 쉽게 표현하는 능력이 뛰어납니다. "data viz" 태그에서 볼 수 있듯이, 데이터 시각화에 대한 자신감과 숙련도를 보여줍니다.

**2. AI/ML 관련성 (3/5)**
"machine learning" 태그가 포함되어 있지만, 이미지 자체만으로는 AI/ML 관련 프로젝트의 구체적인 내용을 파악하기 어렵습니다. 시각화된 데이터가 ML 모델의 결과일 수 있으나, ML 모델 개발이나 복잡한 알고리즘 적용 여부를 직접적으로 판단하기는 어렵습니다. ML 프로젝트에 대한 추가 설명이나 코드가 있다면 평가가 더 높아질 수 있습니다.

**3. 창의성 (3/5)**
전반적으로 깔끔하고 정보를 명확하게 전달하는 데 중점을 둔 시각화 결과물입니다. 호주 지도 위 원형 마커나 연속된 원형 차트 구성 등이 흥미롭습니다. 아주 새롭거나 실험적인 기법보다는 기존 도구를 효과적으로 활용하여 정보를 가공하는 강점이 돋보입니다. 사용된 툴이나 기법에 대한 정보가 있다면 더 깊이 있는 평가가 가능할 것입니다.

---

### 종합 의견:

이지은 지원자는 **데이터 시각화 분야에서 탄탄한 기본기와 숙련도**를 보여줍니다. 복잡한 데이터를 직관적으로 표현하는 능력이 뛰어나 동아리 활동이 데이터 분석 및 시각화와 관련이 있다면 매우 긍정적인 요소입니다. AI/ML 관련성은 이미지로는 깊이를 파악하기 어려우므로, 해당 분야의 프로젝트가 있다면 추가 설명을 통해 역량을 어필하는 것이

## 6. 플러그인 도구 — Essay Analyzer + Google Search (Task 1a 확장)

### 로컬 도구 vs 플러그인 도구

| | 로컬/API 도구 | 플러그인 도구 |
|---|---|---|
| **구현 주체** | 개발자가 Python 함수 작성 | Google 서버에서 이미 구현 |
| **실행 위치** | 클라이언트(내 코드) | Google 서버 |
| **예시** | search_wikipedia (섹션 4) | google_search_retrieval |

### 같은 에이전트, 다른 도구

Essay Analyzer에 Google Search 플러그인을 추가하면 어떻게 달라질까요?
섹션 4a(Wikipedia API) vs 섹션 5(Google Search 플러그인) 결과를 비교해보세요.

> ** 주의**: `google_search_retrieval`(내장 플러그인)과 Python 함수 도구는
> **같은 요청에서 동시 사용 불가**. 새 SDK (`google.genai`)를 사용합니다.

In [9]:
# 새 SDK로 Google Search 플러그인 사용
# Essay Analyzer + Google Search 플러그인
def essay_analyzer_with_search(essay_text: str, applicant_name: str) -> str:
    '''Google Search 플러그인을 활용하는 Essay Analyzer'''
    prompt = f'''당신은 동아리 지원자의 에세이를 심사하는 Essay Analyzer입니다.

지원자: {applicant_name}
자기소개서:
{essay_text}

에세이에서 언급된 기술 용어나 개념을 최신 정보로 검색하여 지원자의 이해도를 평가해주세요.
평가 항목: 기술 이해도(1~5), 동아리 적합성(1~5), 글쓰기 명확성(1~5), 종합 의견'''

    response = client.models.generate_content(
        model='gemini-2.5-flash-lite',
        contents=prompt,
        config=types.GenerateContentConfig(
            tools=[types.Tool(google_search=types.GoogleSearch())]
        )
    )
    return response.text


# 비교: 동일 에세이를 Wikipedia API vs Google Search 플러그인으로 분석
test_essay = '''
저는 대규모 언어 모델(LLM)과 RAG(Retrieval-Augmented Generation)에 관심이 있습니다.
최근 GPT-4와 Gemini 같은 모델들이 에이전트 시스템으로 발전하는 것을 보며
멀티에이전트 프레임워크를 직접 구현해보고 싶어서 지원했습니다.
'''
applicant = '최수진'

print('=== 비교: Wikipedia API vs Google Search 플러그인 ===')

# [방법 1] Wikipedia API 도구 (섹션 4a)
print('\n[방법 1] Essay Analyzer + Wikipedia API')
wiki_chat = client.chats.create(
    model='gemini-2.5-flash-lite',
    config=types.GenerateContentConfig(
        tools=[search_wikipedia],
        system_instruction='동아리 에세이 심사 에이전트입니다. 기술 용어를 Wikipedia로 검색하여 이해도를 평가하세요.'
    )
)
wiki_response = wiki_chat.send_message(f'지원자 {applicant}의 에세이 심사:\n{test_essay}')
print(wiki_response.text[:500])

print('\n' + '-'*55)

# [방법 2] Google Search 플러그인 (섹션 5)
print('\n[방법 2] Essay Analyzer + Google Search 플러그인')
search_result = essay_analyzer_with_search(test_essay, applicant)
print(search_result[:500])

print('\n' + '='*55)
print('💡 차이점 관찰:')
print('   - Wikipedia API: 특정 개념의 정의/요약 정보 제공')
print('   - Google Search: 최신 뉴스/트렌드 포함, 더 넓은 검색 범위')

=== 비교: Wikipedia API vs Google Search 플러그인 ===

[방법 1] Essay Analyzer + Wikipedia API

안녕하세요, 최수진님의 에세이 심사를 돕겠습니다.

먼저, 다음과 같은 기술 용어들에 대한 Wikipedia 검색 결과를 알려드립니다.

*   **대규모 언어 모델(LLM)**: LLM은 수많은 파라미터를 가진 인공 신경망으로, 대량의 텍스트 데이터를 통해 학습됩니다. 2018년경부터 주목받기 시작했으며, 자연어 처리 분야의 다양한 작업에 활용됩니다.

*   **RAG(Retrieval-Augmented Generation)**: 죄송합니다. 'RAG(Retrieval-Augmented Generation)'에 대한 Wikipedia 정보를 현재 찾을 수 없습니다.

*   **멀티에이전트 프레임워크**: 죄송합니다. '멀티에이전트 프레임워크'에 대한 Wikipedia 정보를 현재 찾을 수 없습니다.

혹시 RAG나 멀티에이전트 프레임워크에 대해 더 자세한 설명이나 다른 자료가 있으시다면 알려주시면 감사하겠습니다.

-------------------------------------------------------

[방법 2] Essay Analyzer + Google Search 플러그인
## 자기소개서 에세이 분석 결과

**지원자:** 최수진

**평가 항목:**

1.  **기술 이해도:** 4/5
    *   최수진 지원자는 LLM(Large Language Model)과 RAG(Retrieval-Augmented Generation)에 대한 관심을 명확히 표현하고 있습니다. 최근 GPT-4와 Gemini와 같은 모델들이 에이전트 시스템으로 발전하는 추세를 언급하며 멀티에이전트 프레임워크에 대한 이해를 보여줍니다.
    *   특히 Gemini API를 활용한 에이전트 개발, RAG의 최신 동향, 그리고 멀티 에이전트 프레임워크의 발전 등 관련 기술의 최신 정보들을 파악하고 있음을 알 수

## 7. 메모리 시스템

### 메모리의 3가지 유형

| 유형 | 설명 | 구현 방법 |
|------|------|----------|
| **내부 지식** | 모델이 학습한 지식 | 모델 자체 |
| **단기 메모리** | 현재 대화 컨텍스트 | 대화 히스토리 (in-context) |
| **장기 메모리** | 세션 간 지속 정보 | 외부 DB/파일 |

### 단기 메모리 전략: FIFO vs 요약

Application Collector가 하루에 수십 건의 지원서를 처리한다면 어떻게 대화 히스토리를 관리해야 할까요?

In [10]:
# ① FIFO (First In First Out) 메모리 전략
class FIFOMemory:
    '''가장 오래된 메시지부터 제거하는 단기 메모리'''

    def __init__(self, max_messages: int = 6):
        self.messages = []
        self.max_messages = max_messages

    def add(self, role: str, content: str):
        self.messages.append({'role': role, 'content': content})
        if len(self.messages) > self.max_messages:
            removed = self.messages.pop(0)
            print(f'  ⚠️  FIFO: 오래된 메시지 제거: "{removed["content"][:40]}..."')

    def get_context(self) -> str:
        return '\n'.join([f"{m['role']}: {m['content']}" for m in self.messages])

    def __len__(self):
        return len(self.messages)


# Application Collector 처리 대화 시뮬레이션
print('=== FIFO 메모리 전략 (최대 6개 유지) ===')
fifo_mem = FIFOMemory(max_messages=6)

app_conversations = [
    ('user', '첫 번째 지원자: 이지은, 20240101, 컴퓨터공학과, 파이썬기초'),
    ('assistant', '이지은(20240101) VALID 처리 완료'),
    ('user', '두 번째 지원자: 김철수, 20240202, 물리학과, 선형대수'),
    ('assistant', '김철수(20240202) VALID 처리 완료'),
    ('user', '세 번째 지원자: 박민준, 20240303, 수학과, 파이썬기초,선형대수'),
    ('assistant', '박민준(20240303) VALID 처리 완료'),
    ('user', '첫 번째 지원자 이름이 뭐였지?'),  # FIFO로 초반 메시지 삭제됨
]

for role, content in app_conversations:
    fifo_mem.add(role, content)
    print(f'  [{role}] {content[:60]}')

print(f'\n현재 메모리 크기: {len(fifo_mem)}개')
print('\n현재 컨텍스트:')
print(fifo_mem.get_context())

=== FIFO 메모리 전략 (최대 6개 유지) ===
  [user] 첫 번째 지원자: 이지은, 20240101, 컴퓨터공학과, 파이썬기초
  [assistant] 이지은(20240101) VALID 처리 완료
  [user] 두 번째 지원자: 김철수, 20240202, 물리학과, 선형대수
  [assistant] 김철수(20240202) VALID 처리 완료
  [user] 세 번째 지원자: 박민준, 20240303, 수학과, 파이썬기초,선형대수
  [assistant] 박민준(20240303) VALID 처리 완료
  ⚠️  FIFO: 오래된 메시지 제거: "첫 번째 지원자: 이지은, 20240101, 컴퓨터공학과, 파이썬기초..."
  [user] 첫 번째 지원자 이름이 뭐였지?

현재 메모리 크기: 6개

현재 컨텍스트:
assistant: 이지은(20240101) VALID 처리 완료
user: 두 번째 지원자: 김철수, 20240202, 물리학과, 선형대수
assistant: 김철수(20240202) VALID 처리 완료
user: 세 번째 지원자: 박민준, 20240303, 수학과, 파이썬기초,선형대수
assistant: 박민준(20240303) VALID 처리 완료
user: 첫 번째 지원자 이름이 뭐였지?


In [11]:
# ② 요약 기반 메모리 전략
class SummaryMemory:
    '''대화를 요약하여 중요 정보를 보존하는 메모리'''

    def __init__(self, max_recent: int = 4):
        self.recent_messages = []
        self.summary = ''
        self.max_recent = max_recent

    def add(self, role: str, content: str):
        self.recent_messages.append({'role': role, 'content': content})
        if len(self.recent_messages) > self.max_recent:
            self._summarize_old_messages()

    def _summarize_old_messages(self):
        to_summarize = self.recent_messages[:-self.max_recent//2]
        self.recent_messages = self.recent_messages[-self.max_recent//2:]
        messages_text = '\n'.join([f"{m['role']}: {m['content']}" for m in to_summarize])
        summary_prompt = f'다음 지원서 처리 대화를 핵심 정보(처리된 지원자 이름, 학번, 결과) 위주로 2~3문장으로 요약해줘:\n{messages_text}'
        response = client.models.generate_content(model='gemini-2.5-flash-lite', contents=summary_prompt)
        new_summary = response.text
        self.summary = (self.summary + ' ' + new_summary).strip()
        print(f'  📝 요약 업데이트: "{new_summary[:80]}..."')

    def get_context(self) -> str:
        context = ''
        if self.summary:
            context += f'[이전 처리 요약]\n{self.summary}\n\n'
        context += '[최근 대화]\n'
        context += '\n'.join([f"{m['role']}: {m['content']}" for m in self.recent_messages])
        return context


# 요약 메모리로 동일한 대화 시뮬레이션
print('=== 요약 기반 메모리 전략 ===')
summary_mem = SummaryMemory(max_recent=4)

for role, content in app_conversations[:6]:
    summary_mem.add(role, content)

print('\n현재 컨텍스트:')
print(summary_mem.get_context())

=== 요약 기반 메모리 전략 ===
  📝 요약 업데이트: "첫 번째 지원자인 이지은(학번 20240101)은 컴퓨터공학과 파이썬기초 과목에 대해 VALID 처리가 완료되었습니다. 두 번째 지원자인 김철수..."

현재 컨텍스트:
[이전 처리 요약]
첫 번째 지원자인 이지은(학번 20240101)은 컴퓨터공학과 파이썬기초 과목에 대해 VALID 처리가 완료되었습니다. 두 번째 지원자인 김철수(학번 20240202)는 물리학과 선형대수 과목에 대한 지원을 요청했습니다.

[최근 대화]
assistant: 김철수(20240202) VALID 처리 완료
user: 세 번째 지원자: 박민준, 20240303, 수학과, 파이썬기초,선형대수
assistant: 박민준(20240303) VALID 처리 완료


In [12]:
# ====================================================
# FIFO vs 요약 전략 직접 비교
# ====================================================
print('=' * 60)
print('FIFO vs 요약 메모리 전략 비교')
print('설정: 7턴 대화, 두 전략 모두 최대 4개 메시지 유지')
print('=' * 60)

fifo_mem2 = FIFOMemory(max_messages=4)
summary_mem2 = SummaryMemory(max_recent=4)

print('\n[대화 진행 중...]')
for role, content in app_conversations:
    fifo_mem2.add(role, content)
    summary_mem2.add(role, content)

print('\n' + '-' * 60)
print('📋 FIFO 메모리에 남은 내용 (최근 4개만):')
print(fifo_mem2.get_context())

print('\n' + '-' * 60)
print('📋 요약 메모리에 남은 내용 (요약 + 최근):')
print(summary_mem2.get_context())

# 핵심 테스트: 초반 지원자 이름을 기억하는가?
question = '이 대화에서 처리된 첫 번째 지원자의 이름과 학번이 무엇인지 알 수 있나요? 알 수 없다면 "알 수 없음"으로 답하세요.'

print('\n' + '=' * 60)
print(f'🧪 테스트 질문: "{question}"')
print('=' * 60)

fifo_resp = client.models.generate_content(
    model='gemini-2.5-flash-lite',
    contents=    f'다음은 메모리에 저장된 대화 내용입니다:\n\n{fifo_mem2.get_context()}\n\n질문: {question}'
)
print(f'\n[FIFO 기반 응답] {fifo_resp.text.strip()}')

summary_resp = client.models.generate_content(
    model='gemini-2.5-flash-lite',
    contents=    f'다음은 메모리에 저장된 대화 내용입니다:\n\n{summary_mem2.get_context()}\n\n질문: {question}'
)
print(f'\n[요약 기반 응답] {summary_resp.text.strip()}')

FIFO vs 요약 메모리 전략 비교
설정: 7턴 대화, 두 전략 모두 최대 4개 메시지 유지

[대화 진행 중...]
  ⚠️  FIFO: 오래된 메시지 제거: "첫 번째 지원자: 이지은, 20240101, 컴퓨터공학과, 파이썬기초..."
  📝 요약 업데이트: "이지은(20240101)은 컴퓨터공학과 파이썬기초 지원자로, VALID 처리되었습니다. 김철수(20240202)는 물리학과 선형대수 지원자로, ..."
  ⚠️  FIFO: 오래된 메시지 제거: "이지은(20240101) VALID 처리 완료..."
  ⚠️  FIFO: 오래된 메시지 제거: "두 번째 지원자: 김철수, 20240202, 물리학과, 선형대수..."

------------------------------------------------------------
📋 FIFO 메모리에 남은 내용 (최근 4개만):
assistant: 김철수(20240202) VALID 처리 완료
user: 세 번째 지원자: 박민준, 20240303, 수학과, 파이썬기초,선형대수
assistant: 박민준(20240303) VALID 처리 완료
user: 첫 번째 지원자 이름이 뭐였지?

------------------------------------------------------------
📋 요약 메모리에 남은 내용 (요약 + 최근):
[이전 처리 요약]
이지은(20240101)은 컴퓨터공학과 파이썬기초 지원자로, VALID 처리되었습니다. 김철수(20240202)는 물리학과 선형대수 지원자로, 해당 지원자의 처리 결과는 아직 언급되지 않았습니다.

[최근 대화]
assistant: 김철수(20240202) VALID 처리 완료
user: 세 번째 지원자: 박민준, 20240303, 수학과, 파이썬기초,선형대수
assistant: 박민준(20240303) VALID 처리 완료
user: 첫 번째 지원자 이름이 뭐였지?

🧪 테스트 질문: "이 대화에서 처리된 첫 번째 지원자의 이름과 학번이 무엇인지 알

## 8. 장기 메모리: 세션 간 지원서 처리 기록 유지

Application Collector는 세션이 재시작되어도 이전에 처리한 지원서를 기억해야 합니다.
JSON 파일에 처리된 지원서를 저장하여 **중복 지원을 영구적으로 방지**합니다.

In [13]:
class ApplicationDatabase:
    '''파일 기반 장기 메모리 — 처리된 지원서를 세션 간 유지'''

    def __init__(self, filepath: str = '/tmp/collected_applications.json'):
        self.filepath = filepath
        self.data = self._load()

    def _load(self) -> dict:
        if os.path.exists(self.filepath):
            with open(self.filepath, 'r', encoding='utf-8') as f:
                return json.load(f)
        return {'processed_ids': [], 'applications': []}

    def _save(self):
        with open(self.filepath, 'w', encoding='utf-8') as f:
            json.dump(self.data, f, ensure_ascii=False, indent=2)

    def add_application(self, app_info: dict) -> str:
        '''지원서를 DB에 추가합니다. 중복이면 추가하지 않습니다.'''
        sid = app_info.get('student_id', '')
        if sid in self.data['processed_ids']:
            return f'⚠️  중복: 학번 {sid}는 이미 처리되었습니다.'
        self.data['processed_ids'].append(sid)
        self.data['applications'].append(app_info)
        self._save()
        print(f'  💾 저장: {app_info["name"]} ({sid})')
        return f'✅ 저장 완료: {app_info["name"]} ({sid})'

    def is_duplicate(self, student_id: str) -> bool:
        return student_id in self.data['processed_ids']

    def get_summary(self) -> str:
        count = len(self.data['applications'])
        return f'총 처리된 지원서: {count}건 / 처리된 학번: {self.data["processed_ids"]}'


# 세션 1: 지원서 저장
print('=== 세션 1: 지원서 처리 및 저장 ===')
db = ApplicationDatabase()

new_applications = [
    {'name': '이지은', 'student_id': '20240101', 'major': '컴퓨터공학과', 'courses': ['파이썬 기초']},
    {'name': '박민준', 'student_id': '20240303', 'major': '물리학과', 'courses': ['선형대수']},
    {'name': '이지은', 'student_id': '20240101', 'major': '컴퓨터공학과', 'courses': ['파이썬 기초']},  # 중복
]

for app in new_applications:
    result = db.add_application(app)
    print(f'  결과: {result}')

# 세션 2: 새 세션에서도 이전 기록 유지됨
print('\n=== 세션 2: 새 세션 시작 — 이전 기록 복원 확인 ===')
db2 = ApplicationDatabase()  # 새 인스턴스 = 파일에서 로드
print(db2.get_summary())

# 중복 확인
test_id = '20240101'
is_dup = db2.is_duplicate(test_id)
print(f'\n학번 {test_id} 중복 여부: {"중복" if is_dup else "신규"}')

# 저장된 JSON 확인
print(f'\n📋 저장된 DB 내용:')
with open('/tmp/collected_applications.json', 'r', encoding='utf-8') as f:
    print(json.dumps(json.load(f), ensure_ascii=False, indent=2))

=== 세션 1: 지원서 처리 및 저장 ===
  결과: ⚠️  중복: 학번 20240101는 이미 처리되었습니다.
  결과: ⚠️  중복: 학번 20240303는 이미 처리되었습니다.
  결과: ⚠️  중복: 학번 20240101는 이미 처리되었습니다.

=== 세션 2: 새 세션 시작 — 이전 기록 복원 확인 ===
총 처리된 지원서: 2건 / 처리된 학번: ['20240101', '20240303']

학번 20240101 중복 여부: 중복

📋 저장된 DB 내용:
{
  "processed_ids": [
    "20240101",
    "20240303"
  ],
  "applications": [
    {
      "name": "이지은",
      "student_id": "20240101",
      "major": "컴퓨터공학과",
      "courses": [
        "파이썬 기초"
      ]
    },
    {
      "name": "박민준",
      "student_id": "20240303",
      "major": "물리학과",
      "courses": [
        "선형대수"
      ]
    }
  ]
}


## 9. 실습 과제

### 과제 1: Application Collector 도구 추가
불완전한 지원서를 별도 리스트에 저장하는 `flag_incomplete()` 도구를 구현하세요.

### 과제 2: Essay Analyzer 통합
Wikipedia API 도구 + Google Search 플러그인을 모두 갖춘 Essay Analyzer를 완성하세요.
(힌트: 두 도구를 하나의 에이전트에서 동시 사용할 수 없으므로 라우팅 패턴을 활용하세요)

In [14]:
# 과제 1: flag_incomplete() 도구 구현
incomplete_applications = []  # 불완전한 지원서 저장 리스트

def flag_incomplete(name: str, student_id: str, major: str, courses: str, reason: str) -> str:
    '''필수 정보가 누락된 지원서를 불완전 목록에 저장합니다.

    Args:
        name: 지원자 이름
        student_id: 학번
        major: 전공
        courses: 이수 과목
        reason: 불완전한 이유

    Returns:
        저장 결과 메시지
    '''
    # TODO: 여기에 코드를 작성하세요
    # 힌트 1: incomplete_applications 리스트에 지원서 정보를 딕셔너리로 추가
    # 힌트 2: 'flagged_at' 키에 datetime.now().strftime('%Y-%m-%d %H:%M') 을 저장
    # 힌트 3: f-string으로 저장 결과 메시지 반환
    pass


# 테스트 (구현 완료 후 아래 실행)
# result = flag_incomplete('정보없음', '', '', '', '이름 외 모든 정보 누락')
# print(result)
# print(f'불완전 지원서 목록: {incomplete_applications}')

In [15]:
# 과제 2: 라우팅 패턴으로 Wikipedia API + Google Search 통합
# ⚠️ google_search 플러그인과 Python 함수 도구는 같은 요청에 사용 불가
# → 쿼리 유형에 따라 다른 에이전트로 분기

# Wikipedia 기반 Essay Analyzer (구형 SDK)
_wiki_essay_config = types.GenerateContentConfig(
    tools=[search_wikipedia],
    system_instruction='에세이 심사 에이전트: Wikipedia로 기술 개념 검증 후 평가'
)

def smart_essay_analyzer(essay: str, applicant: str, use_realtime: bool = False) -> str:
    '''쿼리 유형에 따라 Wikipedia API 또는 Google Search를 선택하는 에세이 심사기.

    Args:
        essay: 에세이 내용
        applicant: 지원자 이름
        use_realtime: True이면 Google Search(최신 정보), False이면 Wikipedia(개념 설명)
    '''
    if use_realtime:
        # Google Search 플러그인 사용 (새 SDK)
        print('  [Google Search 플러그인 사용]')
        # TODO: client.models.generate_content()로 Google Search 플러그인 활용
        # 힌트: essay_analyzer_with_search() 함수를 참고하세요 (섹션 5)
        pass
    else:
        # Wikipedia API 도구 사용 (구형 SDK)
        print('  [Wikipedia API 도구 사용]')
        chat = client.chats.create(model='gemini-2.5-flash-lite', config=_wiki_essay_config)
        response = chat.send_message(f'지원자 {applicant}의 에세이 심사:\n{essay}')
        return response.text


# 테스트
test_essay_short = '저는 강화학습과 LLM에 관심이 있어 지원했습니다.'
print('Wikipedia 모드:')
result = smart_essay_analyzer(test_essay_short, '홍길동', use_realtime=False)
if result:
    print(result[:300])

Wikipedia 모드:
  [Wikipedia API 도구 사용]
에세이 주제가 명확하고 좋습니다. 하지만 강화학습과 LLM에 대한 설명이 더 깊이 있으면 좋겠습니다. 두 기술의 연관성이나, 해당 기술을 활용하여 어떤 문제를 해결하고 싶은지에 대한 내용이 추가되면 더욱 흥미로운 에세이가 될 것입니다.

혹시 강화학습과 LLM에 대해 더 자세한 정보를 원하시면 Wikipedia 검색을 도와드릴 수 있습니다.


## 10. 핵심 정리

| 개념 | 핵심 내용 | APD 에이전트 |
|------|----------|--------------|
| **로컬 도구** | Python 함수를 `tools=` 에 직접 전달 | Application Collector (Task 0) |
| **API 기반 도구** | 외부 서비스 HTTP 호출로 능력 확장 | Essay Analyzer + Wikipedia, Portfolio Evaluator + Vision |
| **플러그인 도구** | Google 내장 도구를 이름만으로 활성화 | Essay Analyzer + Google Search (새 SDK) |
| **Tool Calling 흐름** | 질문 → 도구 선택 → 실행 → 결과 통합 → 응답 | 모든 에이전트 공통 |
| **단기 메모리** | FIFO(단순) vs 요약(핵심 보존) 전략 | 지원서 처리 대화 히스토리 관리 |
| **장기 메모리** | JSON 파일로 세션 간 정보 유지 | 처리된 학번 중복 방지 (영구 저장) |

### 도구 유형별 정리

| | 로컬 도구 | API 기반 도구 | 플러그인 도구 |
|--|-----------|--------------|---------------|
| **구현** | Python 함수 직접 작성 | HTTP 요청 함수 작성 | 이름만 전달 |
| **인증** | 불필요 | API Key / OAuth | 모델 API Key |
| **제어** | 완전 제어 | 완전 제어 | Google 서버가 실행 |
| **예시** | validate_fields | search_wikipedia | google_search_retrieval |

---
**다음 exercise 예고**: 단일 에이전트 → 멀티 에이전트 아키텍처 — 여러 에이전트가 협력하여 동아리 모집 파이프라인 전체를 실행

## 11. exercise 3 연계: 클래스 인터페이스로 에이전트 표준화

exercise 3 오케스트레이터가 exercise 2의 에이전트들을 `run()` 메서드로 일관되게 호출할 수 있도록
exercise 1에서 정의한 `BaseAgentWorker`를 상속하여 각 에이전트를 클래스로 래핑합니다.

```
BaseAgentWorker (exercise 1에서 정의)
 ├─ ApplicationCollectorAgent ← Task 0 (로컬 도구)
 ├─ EssayAnalyzerAgent ← Task 1a (Wikipedia API)
 └─ PortfolioEvaluatorAgent ← Task 1c (Gemini Vision)
```

> **참고**: Colab은 노트북 간 상태 공유가 안 되므로, exercise 3 셋업 셀에서 이 클래스들이 간결하게 재정의됩니다.

In [16]:
# ====================================================
# exercise 3 연계: exercise 2 에이전트를 BaseAgentWorker 클래스로 래핑
# ====================================================
# BaseAgentWorker는 exercise 1에서 정의한 기본 클래스입니다.
# 여기서는 동일하게 재정의하여 이 노트북에서도 독립 실행 가능하게 합니다.

class BaseAgentWorker:
    """APD 에이전트 워커의 공통 인터페이스 (exercise 1과 동일)."""
    def __init__(self, name: str, task_id: str):
        self.name = name
        self.task_id = task_id
    def run(self, input_text: str) -> str:
        raise NotImplementedError(f'{self.__class__.__name__}.run()을 구현하세요.')
    def __repr__(self):
        return f'{self.__class__.__name__}(name="{self.name}", task="{self.task_id}")'


# ====================================================
# ApplicationCollectorAgent — Task 0 (로컬 도구)
# ====================================================
class ApplicationCollectorAgent(BaseAgentWorker):
    """exercise 2 Application Collector를 BaseAgentWorker로 래핑합니다.

    validate_fields, check_duplicate, normalize_application 로컬 도구를 사용합니다.
    (위 셀에서 정의된 도구 함수들을 그대로 활용)
    """

    def __init__(self):
        super().__init__(name='Application Collector', task_id='Task 0')
        self._chat = client.chats.create(
            model='gemini-2.5-flash-lite',
            config=types.GenerateContentConfig(
                tools=[validate_fields, check_duplicate, normalize_application],
                system_instruction='''당신은 동아리 지원서를 수집하고 검증하는 Application Collector 에이전트입니다.
지원서를 받으면 반드시 이 순서로 처리하세요:
1. check_duplicate()로 중복 지원 여부 확인
2. validate_fields()로 필드 유효성 검사
3. 유효한 경우 normalize_application()으로 데이터 정규화
각 단계 결과를 명확히 보고하세요.''',
            )
        )

    def run(self, application_text: str) -> str:
        """지원서 텍스트를 받아 수집·검증·정규화 결과를 반환합니다.

        Args:
            application_text: "이름=..., 학번=..., 전공=..., 이수과목=..." 형식의 지원서

        Returns:
            처리 결과 텍스트
        """
        return self._chat.send_message(application_text).text


# ====================================================
# EssayAnalyzerAgent — Task 1a (Wikipedia API 도구)
# ====================================================
class EssayAnalyzerAgent(BaseAgentWorker):
    """exercise 2 Essay Analyzer를 BaseAgentWorker로 래핑합니다.

    search_wikipedia() API 도구를 사용하여 기술 개념을 검증합니다.
    (위 셀에서 정의된 search_wikipedia 함수를 그대로 활용)
    """

    def __init__(self):
        super().__init__(name='Essay Analyzer', task_id='Task 1a')
        self._essay_config = types.GenerateContentConfig(
            tools=[search_wikipedia],
            system_instruction='''당신은 동아리 지원자의 자기소개서를 심사하는 Essay Analyzer 에이전트입니다.
에세이에서 기술 용어를 파악하고, search_wikipedia()로 조회하여 지원자의 이해도를 평가하세요.
평가 항목: 기술 이해도(1~5), 동아리 적합성(1~5), 글쓰기 명확성(1~5), 종합 의견''',
        )

    def run(self, essay_text: str) -> str:
        """에세이 텍스트를 받아 심사 결과를 반환합니다.

        Args:
            essay_text: 지원자 자기소개서 전문

        Returns:
            심사 결과 텍스트 (점수 + 종합 의견)
        """
        chat = client.chats.create(model='gemini-2.5-flash-lite', config=self._essay_config)
        return chat.send_message(f'다음 에세이를 심사해주세요:\n{essay_text}').text


# ====================================================
# PortfolioEvaluatorAgent — Task 1c (Gemini Vision API)
# ====================================================
class PortfolioEvaluatorAgent(BaseAgentWorker):
    """exercise 2 Portfolio Evaluator를 BaseAgentWorker로 래핑합니다.

    analyze_portfolio_image() Gemini Vision 도구를 사용합니다.
    (위 셀에서 정의된 analyze_portfolio_image 함수를 그대로 활용)
    """

    def __init__(self):
        super().__init__(name='Portfolio Evaluator', task_id='Task 1c')
        self._portfolio_config = types.GenerateContentConfig(
            tools=[analyze_portfolio_image],
            system_instruction='''당신은 동아리 지원자의 포트폴리오를 평가하는 Portfolio Evaluator 에이전트입니다.
이미지 URL이 주어지면 analyze_portfolio_image() 도구로 분석하세요.
평가 기준: 기술 수준(1~5), AI/ML 관련성(1~5), 창의성(1~5), 합격 권고 여부''',
        )

    def run(self, image_url: str) -> str:
        """포트폴리오 이미지 URL을 받아 평가 결과를 반환합니다.

        Args:
            image_url: 분석할 포트폴리오 이미지 URL

        Returns:
            포트폴리오 평가 결과 텍스트
        """
        chat = client.chats.create(model='gemini-2.5-flash-lite', config=self._portfolio_config)
        return chat.send_message(f'포트폴리오를 평가해주세요. 이미지 URL: {image_url}').text


# ====================================================
# 테스트: 3개 에이전트 클래스 인터페이스 확인
# ====================================================
print('✅ exercise 2 에이전트 클래스 정의 완료\n')

w2_agents = [
    ApplicationCollectorAgent(),
    EssayAnalyzerAgent(),
    PortfolioEvaluatorAgent(),
]
print('📋 exercise 2 에이전트 목록:')
for agent in w2_agents:
    print(f'  {agent}')

print('\n→ exercise 3에서 이 인터페이스를 통해 APD 전체 파이프라인을 구성합니다.')

✅ exercise 2 에이전트 클래스 정의 완료

📋 exercise 2 에이전트 목록:
  ApplicationCollectorAgent(name="Application Collector", task="Task 0")
  EssayAnalyzerAgent(name="Essay Analyzer", task="Task 1a")
  PortfolioEvaluatorAgent(name="Portfolio Evaluator", task="Task 1c")

→ exercise 3에서 이 인터페이스를 통해 APD 전체 파이프라인을 구성합니다.
